In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import json
import tarfile

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 250)

PROJECT_DIR = Path("..").resolve()

PUBLIC_RAW = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "raw"
)

INTERIM_DIR = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "interim"
)

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [2]:
dictionary = pd.read_csv(
    REPORTS_DIR
    / "feature_dictionary_validation.csv"
)

label_registry = pd.read_csv(
    REPORTS_DIR
    / "dataset_label_registry.csv"
)

catching_targets = pd.read_csv(
    INTERIM_DIR
    / "catching_targets.csv"
)

print(
    "Diccionario:",
    dictionary.shape
)

print(
    "Registro etiquetas:",
    label_registry.shape
)

print(
    "Catching targets:",
    catching_targets.shape
)

Diccionario: (48, 10)
Registro etiquetas: (10, 10)
Catching targets: (30, 8)


In [3]:
x_dictionary = dictionary[
    dictionary["rol_ml"] == "X"
].copy()

print(
    "Variables X:",
    len(x_dictionary)
)

cols_show = [
    col
    for col in [
        "variable",
        "categoria",
        "rol_ml",
        "captura_extension",
        "estado_final"
    ]
    if col in x_dictionary.columns
]

display(
    x_dictionary[
        cols_show
    ]
)

Variables X: 40


,variable,categoria,rol_ml,captura_extension,estado_final
4,task_time_s,TEMPORAL,X,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
5,mean_response_time_ms,TEMPORAL,X,DERIVADA,RECOLECCION_PROPIA
6,max_response_time_ms,TEMPORAL,X,DERIVADA,RECOLECCION_PROPIA
7,mean_pre_action_pause_s,TEMPORAL,X,DERIVADA,RESPALDADA_POR_DATOS_PUBLICOS
8,click_count,INTERACCION,X,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
9,keypress_count,INTERACCION,X,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
10,scroll_count,INTERACCION,X,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
11,backtracking_count,INTERACCION,X,DERIVADA,RESPALDADA_POR_DATOS_PUBLICOS
12,repeated_action_count,INTERACCION,X,DERIVADA,RESPALDADA_POR_DATOS_PUBLICOS
13,navigation_steps,INTERACCION,X,DERIVADA,RESPALDADA_POR_DATOS_PUBLICOS


In [4]:
target_cols = [
    "H1_violation",
    "H4_violation",
    "H5_violation",
    "H6_violation"
]

balance_records = []

for target in target_cols:

    y = pd.to_numeric(
        catching_targets[target],
        errors="coerce"
    )

    n_positive = int(
        (y == 1).sum()
    )

    n_negative = int(
        (y == 0).sum()
    )

    n_missing = int(
        y.isna().sum()
    )

    n_valid = (
        n_positive
        +
        n_negative
    )

    prevalence = (
        n_positive / n_valid
        if n_valid > 0
        else np.nan
    )

    balance_records.append({
        "target": target,
        "positive": n_positive,
        "negative": n_negative,
        "missing": n_missing,
        "valid": n_valid,
        "positive_ratio": prevalence
    })

catching_balance = pd.DataFrame(
    balance_records
)

catching_balance

,target,positive,negative,missing,valid,positive_ratio
0,H1_violation,29,1,0,30,0.966667
1,H4_violation,27,2,1,29,0.931034
2,H5_violation,28,2,0,30,0.933333
3,H6_violation,19,11,0,30,0.633333


In [5]:
source_feasibility = pd.DataFrame([
    {
        "dataset": "AMUSED",
        "X_modality": "BEHAVIOR_INTERACTION",
        "local_X_available": False,
        "X_independent_of_Y": True,
        "Y_type": "HUMAN_PROXY",
        "same_instance_XY": "EXPECTED",
        "canonical_X_alignment": "HIGH_EXPECTED",
        "current_status": "PENDING_ACCESS",
        "next_action": "WAIT_ACCESS"
    },

    {
        "dataset": "Catching UX Flaws",
        "X_modality": "WEB_SOURCE_CODE",
        "local_X_available": False,
        "X_independent_of_Y": True,
        "Y_type": "LLM_DIRECT",
        "same_instance_XY": "POSSIBLE_IF_SOURCE_RECOVERED",
        "canonical_X_alignment": "HIGH_STRUCTURAL",
        "current_status": "LABELS_READY_X_MISSING",
        "next_action": "RECOVER_ORIGINAL_WEBSITE_SOURCE"
    },

    {
        "dataset": "29042675",
        "X_modality": "PROTOTYPE",
        "local_X_available": False,
        "X_independent_of_Y": True,
        "Y_type": "DIRECT_CANDIDATE",
        "same_instance_XY": "POSSIBLE_IF_PROTOTYPE_RECOVERED",
        "canonical_X_alignment": "MEDIUM",
        "current_status": "CONDITIONAL",
        "next_action": "RECOVER_PROTOTYPE"
    },

    {
        "dataset": "SERENE",
        "X_modality": "METRICS",
        "local_X_available": True,
        "X_independent_of_Y": True,
        "Y_type": "REFERENCE_ONLY",
        "same_instance_XY": False,
        "canonical_X_alignment": "UNKNOWN",
        "current_status": "REFERENCE_ONLY",
        "next_action": "DO_NOT_USE_FOR_PRIMARY_TARGET"
    },

    {
        "dataset": "P3M",
        "X_modality": "SCREENSHOT",
        "local_X_available": True,
        "X_independent_of_Y": True,
        "Y_type": "LLM_DIRECT_POSITIVE",
        "same_instance_XY": True,
        "canonical_X_alignment": "LOW_TO_MEDIUM",
        "current_status": "SECONDARY",
        "next_action": "KEEP_AS_VISUAL_REFERENCE"
    },

    {
        "dataset": "PHR",
        "X_modality": "ENERGY_AND_TASK",
        "local_X_available": True,
        "X_independent_of_Y": True,
        "Y_type": "HUMAN_PROXY",
        "same_instance_XY": True,
        "canonical_X_alignment": "LOW",
        "current_status": "PROXY_REFERENCE",
        "next_action": "DO_NOT_MAP_H01_TO_H1_DIRECTLY"
    },

    {
        "dataset": "Dashboards",
        "X_modality": "BEHAVIOR_INTERACTION",
        "local_X_available": True,
        "X_independent_of_Y": True,
        "Y_type": "EXPERIMENTAL_PROXY",
        "same_instance_XY": "JOIN_PENDING",
        "canonical_X_alignment": "HIGH_BEHAVIOR",
        "current_status": "PROMISING_PROXY",
        "next_action": "BUILD_BEHAVIOR_FEATURES"
    },

    {
        "dataset": "UICrit + RICO",
        "X_modality": "UI_HIERARCHY",
        "local_X_available": True,
        "X_independent_of_Y": True,
        "Y_type": "HUMAN_RATING_PROXY",
        "same_instance_XY": True,
        "canonical_X_alignment": "MEDIUM_STRUCTURAL",
        "current_status": "PROMISING_PROXY",
        "next_action": "INSPECT_RICO_HIERARCHIES"
    },

    {
        "dataset": "Zenodo 21891322",
        "X_modality": "SCREENSHOT",
        "local_X_available": True,
        "X_independent_of_Y": True,
        "Y_type": "REFERENCE_ONLY",
        "same_instance_XY": "UNRESOLVED",
        "canonical_X_alignment": "LOW_TO_MEDIUM",
        "current_status": "REFERENCE_ONLY",
        "next_action": "NO_PRIMARY_TRAINING"
    },

    {
        "dataset": "Cultural Heritage",
        "X_modality": None,
        "local_X_available": False,
        "X_independent_of_Y": None,
        "Y_type": "PENDING",
        "same_instance_XY": False,
        "canonical_X_alignment": "UNKNOWN",
        "current_status": "PENDING_DATASET",
        "next_action": "WAIT_DATA"
    }
])

display(source_feasibility)

,dataset,X_modality,local_X_available,X_independent_of_Y,Y_type,same_instance_XY,canonical_X_alignment,current_status,next_action
0,AMUSED,BEHAVIOR_INTERACTION,False,True,HUMAN_PROXY,EXPECTED,HIGH_EXPECTED,PENDING_ACCESS,WAIT_ACCESS
1,Catching UX Flaws,WEB_SOURCE_CODE,False,True,LLM_DIRECT,POSSIBLE_IF_SOURCE_RECOVERED,HIGH_STRUCTURAL,LABELS_READY_X_MISSING,RECOVER_ORIGINAL_WEBSITE_SOURCE
2,29042675,PROTOTYPE,False,True,DIRECT_CANDIDATE,POSSIBLE_IF_PROTOTYPE_RECOVERED,MEDIUM,CONDITIONAL,RECOVER_PROTOTYPE
3,SERENE,METRICS,True,True,REFERENCE_ONLY,False,UNKNOWN,REFERENCE_ONLY,DO_NOT_USE_FOR_PRIMARY_TARGET
4,P3M,SCREENSHOT,True,True,LLM_DIRECT_POSITIVE,True,LOW_TO_MEDIUM,SECONDARY,KEEP_AS_VISUAL_REFERENCE
5,PHR,ENERGY_AND_TASK,True,True,HUMAN_PROXY,True,LOW,PROXY_REFERENCE,DO_NOT_MAP_H01_TO_H1_DIRECTLY
6,Dashboards,BEHAVIOR_INTERACTION,True,True,EXPERIMENTAL_PROXY,JOIN_PENDING,HIGH_BEHAVIOR,PROMISING_PROXY,BUILD_BEHAVIOR_FEATURES
7,UICrit + RICO,UI_HIERARCHY,True,True,HUMAN_RATING_PROXY,True,MEDIUM_STRUCTURAL,PROMISING_PROXY,INSPECT_RICO_HIERARCHIES
8,Zenodo 21891322,SCREENSHOT,True,True,REFERENCE_ONLY,UNRESOLVED,LOW_TO_MEDIUM,REFERENCE_ONLY,NO_PRIMARY_TRAINING
9,Cultural Heritage,NaN,False,None,PENDING,False,UNKNOWN,PENDING_DATASET,WAIT_DATA


In [6]:
CATCHING_DIR = (
    PUBLIC_RAW
    / "04_catching_ux_flaws"
)

catching_files = [
    path
    for path in CATCHING_DIR.rglob("*")
    if path.is_file()
]

source_extensions = {
    ".html",
    ".htm",
    ".css",
    ".js",
    ".jsx",
    ".ts",
    ".tsx",
    ".vue",
    ".php"
}

catching_source_files = [
    path
    for path in catching_files
    if path.suffix.lower()
    in source_extensions
]

print(
    "Archivos totales:",
    len(catching_files)
)

print(
    "Archivos fuente web:",
    len(catching_source_files)
)

display(
    pd.Series(
        [
            path.suffix.lower()
            for path in catching_files
        ]
    )
    .value_counts()
)

Archivos totales: 92
Archivos fuente web: 0


.json     90
.md        1
.ipynb     1
Name: count, dtype: int64

In [7]:
catching_field_policy = pd.DataFrame([
    {
        "field": "site",
        "use_as_X": False,
        "reason": "Identificador/contexto, no predictor."
    },
    {
        "field": "Heuristic",
        "use_as_X": False,
        "reason": "Define directamente el target."
    },
    {
        "field": "IssueFound",
        "use_as_X": False,
        "reason": "Es la etiqueta."
    },
    {
        "field": "SeverityRating",
        "use_as_X": False,
        "reason": "Derivada de la evaluación del problema."
    },
    {
        "field": "IssueDescription",
        "use_as_X": False,
        "reason": "Texto generado durante la evaluación de Y."
    },
    {
        "field": "CodeReference",
        "use_as_X": False,
        "reason": "El evaluador seleccionó el código por estar asociado al problema."
    },
    {
        "field": "CodeSnippet",
        "use_as_X": False,
        "reason": "Alto riesgo de target leakage."
    },
    {
        "field": "EvaluationAnswers",
        "use_as_X": False,
        "reason": "Forma parte del razonamiento usado para producir Y."
    },
    {
        "field": "Recommendation",
        "use_as_X": False,
        "reason": "Se genera después de detectar el problema."
    },
    {
        "field": "original_full_source_code",
        "use_as_X": True,
        "reason": "Puede generar X independientemente de la evaluación."
    }
])

catching_field_policy

,field,use_as_X,reason
0,site,False,"Identificador/contexto, no predictor."
1,Heuristic,False,Define directamente el target.
2,IssueFound,False,Es la etiqueta.
3,SeverityRating,False,Derivada de la evaluación del problema.
4,IssueDescription,False,Texto generado durante la evaluación de Y.
5,CodeReference,False,El evaluador seleccionó el código por estar asociado al problema.
6,CodeSnippet,False,Alto riesgo de target leakage.
7,EvaluationAnswers,False,Forma parte del razonamiento usado para producir Y.
8,Recommendation,False,Se genera después de detectar el problema.
9,original_full_source_code,True,Puede generar X independientemente de la evaluación.


In [8]:
catching_sites = (
    catching_targets[
        ["site"]
    ]
    .drop_duplicates()
    .sort_values("site")
    .reset_index(drop=True)
)

catching_sites[
    "source_recovery_status"
] = "PENDING"

catching_sites[
    "source_path"
] = None

catching_sites[
    "source_url"
] = None

catching_sites

,site,source_recovery_status,source_path,source_url
0,Business-Website,PENDING,None,None
1,Discord-bot-website-template,PENDING,None,None
2,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,PENDING,None,None
3,academic-website-template,PENDING,None,None
4,academicpages.github.io,PENDING,None,None
5,airspace-bootstrap,PENDING,None,None
6,beginner-html-site-styled,PENDING,None,None
7,charity_responsive-template,PENDING,None,None
8,conference-website-template,PENDING,None,None
9,coreui-free-react-admin-template,PENDING,None,None


In [9]:
catching_sites.to_csv(
    REPORTS_DIR
    / "catching_sites_source_recovery.csv",
    index=False
)

In [10]:
dashboards_feature_map = pd.DataFrame([
    {
        "variable": "task_time_s",
        "status": "DERIVABLE",
        "source_signal": "timestampms / Time",
        "notes": "Validar unidad tarea antes de consolidar."
    },
    {
        "variable": "click_count",
        "status": "DERIVABLE_REQUIRES_EVENT_RULE",
        "source_signal": "event + mouse",
        "notes": "Definir exactamente qué eventos representan clic real."
    },
    {
        "variable": "keypress_count",
        "status": "DERIVABLE",
        "source_signal": "event + key",
        "notes": "Contar evento, no contenido escrito."
    },
    {
        "variable": "scroll_count",
        "status": "DERIVABLE_REQUIRES_EPISODE_RULE",
        "source_signal": "event + scroll",
        "notes": "Agrupar scroll continuo en episodios."
    },
    {
        "variable": "mean_pre_action_pause_s",
        "status": "DERIVABLE",
        "source_signal": "timestampms",
        "notes": "Intervalo previo a acción; no interpretar automáticamente como confusión."
    },
    {
        "variable": "repeated_action_count",
        "status": "DERIVABLE_REQUIRES_RULE",
        "source_signal": "event + node + timestampms",
        "notes": "Requiere ventana temporal fija."
    },
    {
        "variable": "navigation_steps",
        "status": "DERIVABLE",
        "source_signal": "url/urlFull sequence",
        "notes": "Contar transiciones de estado URL."
    },
    {
        "variable": "backtracking_count",
        "status": "DERIVABLE",
        "source_signal": "url/urlFull sequence",
        "notes": "Revisitas a estados anteriores."
    },
    {
        "variable": "platform",
        "status": "DERIVABLE",
        "source_signal": "url/urlFull",
        "notes": "Contexto, no predictor principal."
    },
    {
        "variable": "mean_response_time_ms",
        "status": "NO",
        "source_signal": None,
        "notes": "No confundir timestamp entre acciones con Resource Timing."
    },
    {
        "variable": "max_response_time_ms",
        "status": "NO",
        "source_signal": None,
        "notes": "No existe evidencia de Resource Timing."
    }
])

dashboards_feature_map

,variable,status,source_signal,notes
0,task_time_s,DERIVABLE,timestampms / Time,Validar unidad tarea antes de consolidar.
1,click_count,DERIVABLE_REQUIRES_EVENT_RULE,event + mouse,Definir exactamente qué eventos representan clic real.
2,keypress_count,DERIVABLE,event + key,"Contar evento, no contenido escrito."
3,scroll_count,DERIVABLE_REQUIRES_EPISODE_RULE,event + scroll,Agrupar scroll continuo en episodios.
4,mean_pre_action_pause_s,DERIVABLE,timestampms,Intervalo previo a acción; no interpretar automáticamente como confusión.
5,repeated_action_count,DERIVABLE_REQUIRES_RULE,event + node + timestampms,Requiere ventana temporal fija.
6,navigation_steps,DERIVABLE,url/urlFull sequence,Contar transiciones de estado URL.
7,backtracking_count,DERIVABLE,url/urlFull sequence,Revisitas a estados anteriores.
8,platform,DERIVABLE,url/urlFull,"Contexto, no predictor principal."
9,mean_response_time_ms,NO,NaN,No confundir timestamp entre acciones con Resource Timing.


In [11]:
RICO_DIR = (
    PUBLIC_RAW
    / "10_uicrit_rico"
)

uicrit = pd.read_csv(
    RICO_DIR
    / "uicrit_public.csv"
)

ui_details = pd.read_csv(
    RICO_DIR
    / "ui_details.csv"
)

print(
    uicrit.shape,
    ui_details.shape
)

(2981, 9) (66261, 4)


In [12]:
uicrit_field_policy = pd.DataFrame([
    {
        "field": "rico_id",
        "role": "JOIN_KEY",
        "use_as_X": False
    },
    {
        "field": "aesthetics_rating",
        "role": "HUMAN_RATING",
        "use_as_X": False
    },
    {
        "field": "learnability",
        "role": "H6_PROXY_CANDIDATE",
        "use_as_X": False
    },
    {
        "field": "efficency",
        "role": "RATING",
        "use_as_X": False
    },
    {
        "field": "usability_rating",
        "role": "RATING",
        "use_as_X": False
    },
    {
        "field": "design_quality_rating",
        "role": "H4_PROXY_CANDIDATE",
        "use_as_X": False
    },
    {
        "field": "RICO_view_hierarchy",
        "role": "X_SOURCE",
        "use_as_X": True
    }
])

uicrit_field_policy

,field,role,use_as_X
0,rico_id,JOIN_KEY,False
1,aesthetics_rating,HUMAN_RATING,False
2,learnability,H6_PROXY_CANDIDATE,False
3,efficency,RATING,False
4,usability_rating,RATING,False
5,design_quality_rating,H4_PROXY_CANDIDATE,False
6,RICO_view_hierarchy,X_SOURCE,True


In [13]:
feature_maps = []

for _, row in dashboards_feature_map.iterrows():

    feature_maps.append({
        "dataset": "Dashboards",
        "canonical_variable": row["variable"],
        "availability": row["status"],
        "source_signal": row["source_signal"],
        "same_semantics": (
            row["status"] == "DERIVABLE"
        ),
        "notes": row["notes"]
    })

canonical_feature_map = pd.DataFrame(
    feature_maps
)

display(canonical_feature_map)

,dataset,canonical_variable,availability,source_signal,same_semantics,notes
0,Dashboards,task_time_s,DERIVABLE,timestampms / Time,True,Validar unidad tarea antes de consolidar.
1,Dashboards,click_count,DERIVABLE_REQUIRES_EVENT_RULE,event + mouse,False,Definir exactamente qué eventos representan clic real.
2,Dashboards,keypress_count,DERIVABLE,event + key,True,"Contar evento, no contenido escrito."
3,Dashboards,scroll_count,DERIVABLE_REQUIRES_EPISODE_RULE,event + scroll,False,Agrupar scroll continuo en episodios.
4,Dashboards,mean_pre_action_pause_s,DERIVABLE,timestampms,True,Intervalo previo a acción; no interpretar automáticamente como confusión.
5,Dashboards,repeated_action_count,DERIVABLE_REQUIRES_RULE,event + node + timestampms,False,Requiere ventana temporal fija.
6,Dashboards,navigation_steps,DERIVABLE,url/urlFull sequence,True,Contar transiciones de estado URL.
7,Dashboards,backtracking_count,DERIVABLE,url/urlFull sequence,True,Revisitas a estados anteriores.
8,Dashboards,platform,DERIVABLE,url/urlFull,True,"Contexto, no predictor principal."
9,Dashboards,mean_response_time_ms,NO,NaN,False,No confundir timestamp entre acciones con Resource Timing.


In [14]:
training_requirements = pd.DataFrame([
    {
        "requirement": "X_AND_Y_SAME_INSTANCE",
        "required": True
    },
    {
        "requirement": "X_INDEPENDENT_OF_Y_CREATION",
        "required": True
    },
    {
        "requirement": "TARGET_PROVENANCE_RECORDED",
        "required": True
    },
    {
        "requirement": "CANONICAL_FEATURE_SEMANTICS_COMPATIBLE",
        "required": True
    },
    {
        "requirement": "NO_TARGET_LEAKAGE",
        "required": True
    },
    {
        "requirement": "SOURCE_DATASET_RECORDED",
        "required": True
    }
])

training_requirements

,requirement,required
0,X_AND_Y_SAME_INSTANCE,True
1,X_INDEPENDENT_OF_Y_CREATION,True
2,TARGET_PROVENANCE_RECORDED,True
3,CANONICAL_FEATURE_SEMANTICS_COMPATIBLE,True
4,NO_TARGET_LEAKAGE,True
5,SOURCE_DATASET_RECORDED,True


In [15]:
catching_balance.to_csv(
    REPORTS_DIR
    / "catching_target_balance.csv",
    index=False
)

source_feasibility.to_csv(
    REPORTS_DIR
    / "source_feature_feasibility.csv",
    index=False
)

catching_field_policy.to_csv(
    REPORTS_DIR
    / "catching_field_policy.csv",
    index=False
)

dashboards_feature_map.to_csv(
    REPORTS_DIR
    / "dashboards_feature_map.csv",
    index=False
)

uicrit_field_policy.to_csv(
    REPORTS_DIR
    / "uicrit_field_policy.csv",
    index=False
)

canonical_feature_map.to_csv(
    REPORTS_DIR
    / "canonical_feature_map_initial.csv",
    index=False
)

training_requirements.to_csv(
    REPORTS_DIR
    / "training_dataset_requirements.csv",
    index=False
)

In [16]:
expected = [
    "catching_target_balance.csv",
    "source_feature_feasibility.csv",
    "catching_field_policy.csv",
    "catching_sites_source_recovery.csv",
    "dashboards_feature_map.csv",
    "uicrit_field_policy.csv",
    "canonical_feature_map_initial.csv",
    "training_dataset_requirements.csv"
]

for filename in expected:

    path = (
        REPORTS_DIR
        / filename
    )

    print(
        filename,
        "✅" if path.exists() else "❌"
    )

catching_target_balance.csv ✅
source_feature_feasibility.csv ✅
catching_field_policy.csv ✅
catching_sites_source_recovery.csv ✅
dashboards_feature_map.csv ✅
uicrit_field_policy.csv ✅
canonical_feature_map_initial.csv ✅
training_dataset_requirements.csv ✅
